# ResearchLanka Kaggle Main-Branch Full Run

Import this notebook into Kaggle and run cells from top to bottom.

It will:

- clone/pull the latest `main` branch
- copy your uploaded raw dataset into the repo
- install Python + Dagster dependencies
- run the Dagster no-collection preprocessing job
- build best-quality embeddings
- train Logistic Regression
- train Multinomial Naive Bayes baseline
- train Linear SVM
- run formal model comparison and promote the best flat classifier
- train hierarchical field -> subfield Linear SVM
- run NMF topic modeling and evaluation
- zip outputs for download

It does **not** collect data from APIs or repositories.


## 1. Settings

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/krish-anu/researchlanka-ai.git"
BRANCH = "main"
WORK_DIR = Path("/kaggle/working")
CODE_DIR = WORK_DIR / "code"
BACKEND_DIR = CODE_DIR / "backend"
DATASET_DATA_DIR = Path("/kaggle/input/datasets/anusankrishnathas/researchlanka-raw-data/backend/data")
OUTPUT_ZIP = WORK_DIR / "researchlanka-kaggle-outputs.zip"

print("Repo:", REPO_URL)
print("Branch:", BRANCH)
print("Code dir:", CODE_DIR)
print("Backend dir:", BACKEND_DIR)
print("Dataset data dir:", DATASET_DATA_DIR)
print("Output zip:", OUTPUT_ZIP)

## 2. Check Kaggle Dataset Exists

If this fails, your Kaggle dataset path is different. Update `DATASET_DATA_DIR` above.

In [ ]:
!ls -la /kaggle/input
!find /kaggle/input -maxdepth 5 -type d | head -80
!test -d {DATASET_DATA_DIR} && echo "Dataset path OK" || echo "Dataset path NOT FOUND"

## 3. Clone Or Pull Latest Main Branch

In [ ]:
%cd /kaggle/working
if CODE_DIR.exists() and (CODE_DIR / ".git").exists():
    %cd /kaggle/working/code
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} code
    %cd /kaggle/working/code

!git log --oneline -3
!ls

## 4. Copy Uploaded Raw Data Into Backend

In [ ]:
%cd /kaggle/working/code/backend
!rm -rf data
!mkdir -p data
!cp -r {DATASET_DATA_DIR}/* data/
!find data -maxdepth 3 -type f | head -60

## 5. Install Dependencies

Kaggle may show dependency conflict warnings. Continue if the install completes. If the session restarts, rerun from the top.

In [ ]:
%cd /kaggle/working/code/backend
!pip install $(grep -v '^psycopg2==' requirements.txt)
!pip install dagster==1.13.16 dagster-webserver
!pip install -e dagster-quickstart
!python -m dagster --version

## 6. Run Dagster Pipeline Without Data Collection

This prepares existing source files and runs preprocessing through the analysis-ready dataset.

In [ ]:
%cd /kaggle/working/code/backend/dagster-quickstart
!python -m dagster job execute \
  -m dagster_quickstart.definitions \
  -j researchlanka_no_collection_preprocessing_job

## 7. Verify Preprocessing Outputs

In [ ]:
%cd /kaggle/working/code/backend
!ls -lh data/processed/repositories_combined.csv
!ls -lh data/processed/sljol.csv
!ls -lh data/processed/common/common_publications_final.csv
!ls -lh data/processed/common/common_publications_final_2016_2026_analysis_ready.csv

import pandas as pd
paths = [
    'data/processed/common/common_publications_final.csv',
    'data/processed/common/common_publications_final_2016_2026_analysis_ready.csv',
]
for path in paths:
    frame = pd.read_csv(path, nrows=5)
    total = sum(1 for _ in open(path, encoding='utf-8')) - 1
    print(path, 'rows=', total, 'columns=', len(frame.columns))


## 8. Build Best-Quality Embeddings

Full text fields, trigrams, larger vocabulary, 512 dimensions, no row limit.

In [ ]:
%cd /kaggle/working/code/backend
!make model-embeddings PYTHON=python \
  EMBED_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  EMBED_MAX_FEATURES=100000 \
  EMBED_NGRAM_MAX=3 \
  EMBED_DIM=512
!ls -lh data/models/publication_text_embeddings.parquet data/models/publication_text_embedding_model.joblib data/models/publication_text_embeddings_summary.txt

## 9. Train Best-Quality Logistic Regression

In [ ]:
%cd /kaggle/working/code/backend
!make train-logreg PYTHON=python \
  LOGREG_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  LOGREG_MAX_FEATURES=100000 \
  LOGREG_NGRAM_MAX=3 \
  LOGREG_MAX_ITER=2000
!cat data/models/logistic_regression_primary_domain_metrics.txt

In [ ]:
%cd /kaggle/working/code/backend
!pip install -e . --no-deps

## 10. Train Best-Quality Linear SVM

If Kaggle RAM fails, rerun this cell with `--max-features 50000`.

In [ ]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_classifier.py \
  --input data/processed/common/common_publications_final.csv \
  --label-column primary_domain \
  --text-columns title,abstract,topics,keywords,concepts \
  --ngram-max 3 \
  --max-features 100000 \
  --c-values 0.1,1,10 \
  --cv-folds 3 \
  --class-weight balanced \
  --max-iter 5000
!cat data/models/linear_svm_primary_domain_metrics.txt

## 11. Train Naive Bayes Baseline


In [ ]:
%cd /kaggle/working/code/backend
!make train-nb PYTHON=python \
  NB_INPUT=data/processed/common/common_publications_final.csv \
  NB_LABEL_COLUMN=primary_domain \
  NB_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  NB_ALPHA=1.0 \
  NB_MIN_CLASS_COUNT=20 \
  NB_TEST_SIZE=0.2
!cat data/models/multinomial_nb_primary_domain_metrics.txt


## 12. Formal Flat Classifier Comparison

This retrains Logistic Regression and Linear SVM with the same data settings, ranks by macro F1, and copies the winner into `data/models/final/`.


In [ ]:
%cd /kaggle/working/code/backend
!python scripts/modeling/compare_classification_models.py \
  --input data/processed/common/common_publications_final.csv \
  --label-column primary_domain \
  --text-columns title,abstract,topics,keywords,concepts \
  --max-features 100000 \
  --ngram-max 3 \
  --c-values 0.1,1,10 \
  --cv-folds 3 \
  --class-weight balanced \
  --ranking-metric macro_f1

!cat data/models/classification_comparison/model_comparison.csv
!ls -lh data/models/final


## 13. Evaluate Prediction Files Together


In [ ]:
%cd /kaggle/working/code/backend
!make evaluate-models PYTHON=python \
  EVAL_PREDICTIONS="--predictions-csv data/models/multinomial_nb_primary_domain_predictions.csv --predictions-csv data/models/logistic_regression_primary_domain_predictions.csv --predictions-csv data/models/linear_svm_primary_domain_predictions.csv" \
  EVAL_OUTPUT_DIR=data/models/evaluation
!find data/models/evaluation -maxdepth 2 -type f -print | sort


## 14. Train Hierarchical Field -> Subfield Linear SVM


In [ ]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_hierarchical.py \
  --input data/processed/common/common_publications_final.csv \
  --field-column primary_field \
  --subfield-column primary_subfield \
  --text-columns title,abstract,topics,keywords,concepts \
  --max-features 100000 \
  --ngram-max 3 \
  --c-value 1.0 \
  --class-weight balanced \
  --max-iter 5000 \
  --field-model-output data/models/linear_svm_hierarchical_field.joblib \
  --subfield-model-output data/models/linear_svm_hierarchical_subfields.joblib \
  --metrics-output data/models/linear_svm_hierarchical_metrics.txt \
  --label-counts-output data/models/linear_svm_hierarchical_labels.csv \
  --manifest-output data/models/linear_svm_hierarchical_manifest.json \
  --predict-output data/models/linear_svm_hierarchical_predictions.csv
!cat data/models/linear_svm_hierarchical_metrics.txt
!ls -lh data/models/linear_svm_hierarchical*


## 15. Run NMF Topic Modeling And Evaluation


In [ ]:
%cd /kaggle/working/code/backend
!python scripts/modeling/run_nmf_topic_modeling.py \
  --data data/processed/common/common_publications_final.csv \
  --output-dir data/processed/common/nmf \
  --k-range 8 10 12 15 20 \
  --n-words 15 \
  --naming-words 3 \
  --text-columns title abstract topics keywords concepts
!find data/processed/common/nmf -maxdepth 1 -type f -print | sort
!cat data/processed/common/nmf/nmf_k_sweep_evaluation.csv


## 16. Verify All Modeling Outputs


In [ ]:
%cd /kaggle/working/code/backend
import shutil
from pathlib import Path

aliases = {
    "data/models/linear_svm_hierarchical_subfield.joblib": "data/models/linear_svm_hierarchical_subfields.joblib",
    "data/models/linear_svm_hierarchical_field_metrics.txt": "data/models/linear_svm_hierarchical_metrics.txt",
}
for source, target in aliases.items():
    source_path = Path(source)
    target_path = Path(target)
    if source_path.exists() and not target_path.exists():
        target_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, target_path)
        print(f"Aliased {source} -> {target}")

required = [
    "data/models/publication_text_embeddings.parquet",
    "data/models/publication_text_embedding_model.joblib",
    "data/models/logistic_regression_primary_domain.joblib",
    "data/models/logistic_regression_primary_domain_metrics.txt",
    "data/models/multinomial_nb_primary_domain.joblib",
    "data/models/multinomial_nb_primary_domain_metrics.txt",
    "data/models/linear_svm_primary_domain.joblib",
    "data/models/linear_svm_primary_domain_metrics.txt",
    "data/models/classification_comparison/model_comparison.csv",
    "data/models/final/publication_field_classifier.joblib",
    "data/models/linear_svm_hierarchical_field.joblib",
    "data/models/linear_svm_hierarchical_subfields.joblib",
    "data/models/linear_svm_hierarchical_metrics.txt",
    "data/processed/common/nmf/nmf_k_sweep_evaluation.csv",
]

missing = [p for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing modeling outputs:\n" + "\n".join(missing))

for p in required:
    path = Path(p)
    print(f"OK {p} ({path.stat().st_size / (1024*1024):.2f} MB)")


## 17. Zip Outputs For Download


In [ ]:
%cd /kaggle/working/code/backend
!rm -f /kaggle/working/researchlanka-kaggle-outputs.zip
!zip -r /kaggle/working/researchlanka-kaggle-outputs.zip data/processed data/models
!ls -lh /kaggle/working/researchlanka-kaggle-outputs.zip


Download these files from the Kaggle output panel:


In [ ]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/researchlanka-kaggle-outputs.zip"))


## 18. Optional: Create Models-Only Zip


In [ ]:
import os
import zipfile
from IPython.display import FileLink, display

source_zip = "/kaggle/working/researchlanka-kaggle-outputs.zip"
models_zip = "/kaggle/working/researchlanka-models-only.zip"

with zipfile.ZipFile(source_zip, "r") as src:
    model_files = [
        name for name in src.namelist()
        if name.startswith("data/models/") and not name.endswith("/")
    ]
    print("Model files found:", len(model_files))
    with zipfile.ZipFile(models_zip, "w", zipfile.ZIP_DEFLATED) as dst:
        for name in model_files:
            dst.writestr(name, src.read(name))

print("Created:", models_zip)
print("Size MB:", round(os.path.getsize(models_zip) / (1024**2), 2))
display(FileLink(models_zip))
